### Imports

In [ ]:
import pandas as pd
import ta
import yfinance as yf
import alpaca_trade_api as trade_api

### Get stock data

**Currently a test**

In [1]:
ticker = "AAPL"
data = yf.download(ticker, period="3mo", interval="1d")

# fix close data typing issue
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
print(type(data['Close']))
print(data['Close'].shape)
print(data.tail())

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.series.Series'>
(62,)
Price            Close        High         Low        Open    Volume
Date                                                                
2026-04-24  271.059998  273.059998  269.649994  272.760010  38157100
2026-04-27  267.609985  268.359985  265.070007  266.089996  41466800
2026-04-28  270.709991  273.230011  268.660004  272.339996  40018900
2026-04-29  270.170013  271.040009  267.040009  267.549988  30047900
2026-04-30  271.350006  275.940002  268.140015  270.500000  69274050


### Adding indicators

Right now I am using relative strength index (RSI). RSI is the relative strength index from 0-100. 30- = oversold (price drop), 70+ = overbought. I'm using RSI to detect potential rebounds after drops

Maybe I could use other indicators such as moving average, multiples such as Price to earnings, price to book value, + price to free cash flow

In [ ]:
data['rsi'] = ta.momentum.RSIIndicator(data['Close'], window=14).rsi()
data['rsi']

Date
2026-02-02          NaN
2026-02-03          NaN
2026-02-04          NaN
2026-02-05          NaN
2026-02-06          NaN
                ...    
2026-04-24    59.671863
2026-04-27    55.161877
2026-04-28    58.217676
2026-04-29    57.482801
2026-04-30    58.709387
Name: rsi, Length: 62, dtype: float64

### Detecting drops in price

In [ ]:
data['pct_change'] = data['Close'].pct_change() * 100

def check_entry(row):
    return row['pct_change'] <= -3 and row['rsi'] < 30
data['buy_signal'] = (data['pct_change'] <= -3) & (data['rsi'] < 30)

Date
2026-02-02         NaN
2026-02-03   -0.196291
2026-02-04    2.601295
2026-02-05   -0.209765
2026-02-06    0.800979
                ...   
2026-04-24   -0.866765
2026-04-27   -1.272785
2026-04-28    1.158404
2026-04-29   -0.199467
2026-04-30    0.436759
Name: pct_change, Length: 62, dtype: float64

### Demonstrativ backtesting

BUY CONDITIONS:
- Buy signal present, no curent position
then uses all money t buy shares

SELL CONDITIONS:
- take profit @ +2%
- Stop loss @ -1.5%

In [ ]:
balance = 10000
position = 0

for i in range(len(data)):
    row = data.iloc[i]

    if row['buy_signal'] and position == 0:
        entry_price = row['Close']
        position = balance / entry_price
        balance = 0
        print(f"BUY at {entry_price}")
    
    elif position > 0:
        current_price = row['Close']
        change = (current_price - entry_price) / entry_price

        if change >= 0.02 or change <= -0.015:
            balance = position * current_price
            position = 0
            print(f"SELL at {current_price}")